# 07. INSERT Statement

*Module: 3 — DML · Companion summary: `lectures/07-insert-statement.md` · SQL engine: SQLite (in-memory)*

## What you will study

- write the INSERT statement: table, column list, VALUES
- insert one row at a time and several rows in one statement
- keep value-count = column-count, always

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 One row at a time — with the full column list

In [ ]:
q("CREATE TABLE author (author_id TEXT PRIMARY KEY, lastname TEXT NOT NULL, firstname TEXT NOT NULL, email TEXT, city TEXT, country TEXT)")
q("""
INSERT INTO author (author_id, lastname, firstname, email, city, country)
VALUES ('A1', 'Chong', 'Raul', 'RFC@ca.ibm.com', 'Toronto', 'CA')
""")
q("SELECT * FROM author")

## 🧪 Multiple rows in ONE statement (the course's second method)

Each row is its own `( ... )`; rows are separated by commas:

In [ ]:
q("""
INSERT INTO author (author_id, lastname, firstname, email, city, country) VALUES
    ('A2', 'Ahuja', 'Rav',  'RA@ca.ibm.com', 'Markham',  'CA'),
    ('A3', 'Katta', 'Lakshmi', 'LK@in.ibm.com', 'Bengaluru', 'IN')
""")
q("SELECT * FROM author")

## 🧪 The golden rule, violated live

In [ ]:
# Some course-syntax errors are lessons themselves — run and read the error.
import sqlite3
try:
    conn.execute("INSERT INTO author (author_id, lastname, firstname) VALUES ('A9', 'Solo')")
except sqlite3.Error as e:
    print(repr(e))   # 2 columns in the list but 3 values given → refused

## 💡 Tips & tricks

- **Always name your columns** — positional INSERTs break when the schema changes.
- Text values need quotes ('Chong'); numbers don't. 'A1' is a string because the column
  is TEXT/CHAR.
- 📕 Note: SQLite additionally supports `INSERT OR IGNORE` / `INSERT OR REPLACE` for
  constraint collisions — handy for study loops.

## 🎯 Your turn

1. Insert two more authors (your own data!) in one multi-row statement.
2. Insert a row into a NEW table `publisher` you invent with 3 columns.

In [ ]:
# 💡 Solution
q("""
INSERT INTO author (author_id, lastname, firstname, email, city, country) VALUES
    ('A4', 'Ali', 'Nadia', 'NA@eg.ibm.com', 'Cairo', 'EG'),
    ('A5', 'Roux', 'Claire', 'CR@fr.ibm.com', 'Lyon', 'FR')
""")
q("SELECT * FROM author")
q("CREATE TABLE publisher (pub_id TEXT PRIMARY KEY, name TEXT NOT NULL, founded INTEGER)")
q("INSERT INTO publisher (pub_id, name, founded) VALUES ('P1', 'OReilly-Demo', 1978)")
q("SELECT * FROM publisher")